# LLM-as-a-Judge for PersonaMem QA

Evaluate model answers using an LLM judge. We have rich per-QA data: question, predicted answer, correct/incorrect answers, **retrieved memories**, persona, and metadata.

## Evaluation dimensions (what to score?)

| Dimension | Inputs | What it measures |
|-----------|--------|------------------|
| **Answer correctness** (minimal) | question, correct_answer, pred_answer | Does the model's answer match the intended answer (semantically)? |
| **Answer correctness + distractors** | + incorrect_answers | Is the pred closer to correct than to known-bad answers? Reduces false positives. |
| **Retrieval relevance** | question, retrieved_memories | Were the right memories retrieved for this question? (Useful for retrieval ablations.) |
| **Grounding** | question, retrieved_memories, pred_answer | Is the answer grounded in the retrieved memories (vs hallucination)? |
| **Persona/style** (optional) | short_persona, pred_answer | Does the answer match the intended persona tone? |

**Recommendation:** Start with **answer correctness** (question + correct + pred, optionally + incorrect_answers). Add **retrieval relevance** and **grounding** if you care about debugging retrieval vs generation, or for ablations (e.g. different memory counts).

### One judge call vs separate calls per dimension

**Use separate judge calls per dimension** (one API call for correctness, one for grounding, one for retrieval_relevance if you use them). Reasons:
- **Clear metrics**: You get a distinct score per dimension (e.g. correctness vs grounding), so you can see where the system fails.
- **Simpler prompts**: Each prompt has a single task; the judge is less likely to mix criteria or output ambiguous scores.
- **Easier parsing**: One structured output per call (e.g. `{"score": 0|1, "reason": "..."}`).
- **Flexibility**: You can run only correctness on the full set, and grounding/relevance on a subset or only when debugging.

Doing "all in one go" (one prompt asking for correctness + grounding + relevance) is possible but: the prompt gets long, the output format becomes multi-part and harder to parse, and the model may conflate criteria (e.g. give a single overall score instead of three).

# Reading QA Results

In [1]:
import json

In [2]:
def load_qa_results(qa_json_path):
    with open(qa_json_path, 'r') as f:
        return json.load(f)


In [3]:
results = load_qa_results("benchmark_logs/multi_gpt-4.1.-mini_graph-img-preserved-2/stage2_RUN-3_qa_20260312_112818.json")

In [4]:
results.keys()

dict_keys(['stage', 'jsonl_path', 'num_qa_pairs', 'total_wall_seconds', 'total_search_calls', 'total_images_processed', 'images_processed_per_user', 'total_input_tokens', 'total_output_tokens', 'total_answer_llm_calls', 'total_answer_input_tokens', 'total_answer_output_tokens', 'total_answer_wall_seconds', 'per_qa', 'errors', 'config', 'mem0_version', 'openai_version', 'summary'])

In [5]:
len(results['per_qa'])

5000

In [6]:
results['per_qa'][0].keys()

dict_keys(['user_id', 'row_index', 'question_id', 'user_query', 'wall_seconds', 'success', 'user_images_processed', 'error', 'search_calls', 'num_results', 'retrieved_memories', 'mem0_answer', 'answer_llm_calls', 'answer_input_tokens', 'answer_output_tokens', 'answer_error', 'answer_wall_seconds', 'correct_answer', 'incorrect_answers', 'short_persona', 'expanded_persona', 'preference', 'topic_preference', 'pref_type', 'sensitive_info', 'conversation_scenario', 'topic_query', 'input_tokens', 'output_tokens', 'used_graph', 'graph_error'])

## Next steps: calling the judge and aggregating

1. **Call your judge LLM** with the prompt (e.g. OpenAI `chat.completions`, or another model). Ask for structured output: e.g. `{"score": 0|1, "reason": "..."}` so you can parse reliably.
2. **Run over a sample or full `per_qa`**: build the prompt with `evaluate_with_llm_as_a_judge(...)`, call the judge, append `eval_result` (or a separate list) with `score` and `reason`.
3. **Aggregate**: accuracy = mean(score) over evaluated items. Optionally slice by `pref_type`, `topic_query`, or `num_results` to see where the system fails.
4. **Optional dimensions**: Run a second pass with `mode="grounding"` to see if good retrieval leads to grounded answers; run `mode="retrieval_relevance"` to score retrieval quality independently.

In [7]:
LLM_AS_A_JUDGE_PROMPT = """
Your task is to label an answer to a question as "CORRECT" or "WRONG". You will be given the following data: (1) a question (posed by one user to another user), (2) a ‘gold’ (ground truth) answer, (3) a generated answer which you will score as CORRECT/WRONG.

The point of the question is to ask about something one user should know about the other user based on their prior conversations. The gold answer will usually be a concise and short answer that includes the referenced topic, for example:

Question: Do you remember what I got the last time I went to Hawaii?
Gold answer: A shell necklace

The generated answer might be much longer, but you should be generous with your grading - as long as it touches on the same topic as the gold answer, it should be counted as CORRECT.

For time related questions, the gold answer will be a specific date, month, year, etc. The generated answer might be much longer or use relative time references (like ‘last Tuesday’ or ‘next month’), but you should be generous with your grading - as long as it refers to the same date or time period as the gold answer, it should be counted as CORRECT. Even if the format differs (e.g., ‘May 7th’ vs ‘7 May’), consider it CORRECT if it’s the same date.

Now it’s time for the real question:
Question: {question}
Gold answer: {gold_answer}
Generated answer: {generated_answer}

First, provide a short (one sentence) explanation of your reasoning, then finish with CORRECT or WRONG. Do NOT include both CORRECT and WRONG in your response, or it will break the evaluation script.

Just return the label CORRECT or WRONG in a json format with the key as "label".
"""

In [8]:
import os
from openai import OpenAI

In [9]:
def build_judge_prompt(question: str, gold_answer: str, generated_answer: str) -> str:
    return LLM_AS_A_JUDGE_PROMPT.format(
        question=question,
        gold_answer=gold_answer,
        generated_answer=generated_answer,
    )


def _extract_label(raw_text: str) -> dict:
    text = (raw_text or "").strip()
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            label = str(parsed.get("label", "")).strip().upper().rstrip(".")
            if label in {"CORRECT", "WRONG"}:
                return {"label": label}
    except Exception:
        pass

    upper = text.upper()
    if "CORRECT" in upper and "WRONG" not in upper:
        return {"label": "CORRECT"}
    if "WRONG" in upper and "CORRECT" not in upper:
        return {"label": "WRONG"}
    return {"error": "Invalid JSON returned", "raw": text[:300]}


def evaluate_with_llm_as_a_judge(question, answer, correct_answer) -> dict:
    # Correct ordering: gold=correct_answer, generated=answer
    prompt = build_judge_prompt(
        question=question,
        gold_answer=correct_answer,
        generated_answer=answer,
    )

    client = OpenAI(
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url="https://openrouter.ai/api/v1",
    )

    completion = client.chat.completions.create(
        model="google/gemini-2.5-flash",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100,
        response_format={"type": "json_object"},
    )

    raw = completion.choices[0].message.content.strip()
    return _extract_label(raw)

In [10]:
print("Key loaded:", os.getenv("OPENROUTER_API_KEY")[:10])

Key loaded: sk-or-v1-a


In [11]:
from tqdm import tqdm

llm_results = []
for qa in tqdm(results["per_qa"], total=len(results["per_qa"]), desc="LLM judging"):
    question = qa["user_query"]
    answer = qa["mem0_answer"]
    correct_answer = qa["correct_answer"]

    result = evaluate_with_llm_as_a_judge(question, answer, correct_answer)

    qa["is_correct"] = result.get('label')

    llm_results.append(result)

LLM judging: 100%|██████████| 5000/5000 [1:09:59<00:00,  1.19it/s]


In [13]:
llm_results[0]

{'label': 'WRONG'}

In [19]:
sum([1 if r.get('label') == "CORRECT" else 0 for r in llm_results]) / len(llm_results) * 100

53.72

In [14]:
with open("benchmark_logs/multi_gpt-4.1.-mini_graph-img-preserved-2/checked_results.json", "w") as f:
    json.dump(results, f, indent=4)